### Required Discussion 19:1: Building a Recommender System with SURPRISE

This discussion focuses on exploring additional algorithms with the `Suprise` library to generate recommendations.  Your goal is to identify the optimal algorithm by minimizing the mean squared error using cross validation. You are also going to select a dataset to use from [grouplens](https://grouplens.org/datasets/movielens/) example datasets.  

To begin, head over to [grouplens](https://grouplens.org/datasets/movielens/) and examine the different datasets available.  Choose one so that it is easy to create the data as expected in `Surprise` with user, item, and rating information.  Then, compare the performance of at least the `KNNBasic`, `SVD`, `NMF`, `SlopeOne`, and `CoClustering` algorithms to build your recommendations.  For more information on the algorithms see the documentation for the algorithm package [here](https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html).

Share the results of your investigation and include the results of your cross validation and a basic description of your dataset with your peers.


In [2]:
# 1. Uninstall the existing NumPy and Surprise
!pip uninstall -y numpy scikit-surprise

# 2. Install a compatible NumPy (1.26.4 is the stable 1.x bridge) and Surprise
!pip install numpy==1.26.4 scikit-surprise

# 3. Force a session restart (this clears the old NumPy from memory)
import os
os._exit(0)

In [1]:
import numpy
import surprise
print(f"NumPy version: {numpy.__version__}")
print("Surprise imported successfully!")

NumPy version: 1.26.4
Surprise imported successfully!


In [ ]:
from surprise import Dataset, Reader, SVD, NMF, KNNBasic, SlopeOne, CoClustering
from surprise.model_selection import cross_validate

import pandas as pd

In [5]:
movie_reviews = pd.read_csv("/content/sample_data/ml-surprise-ratings.csv")
movie_reviews.head(2)

In [17]:
movie_reviews["rating"].unique()

array([4. , 5. , 3. , 2. , 1. , 4.5, 3.5, 2.5, 0.5, 1.5])

Choose one so that it is easy to create the data as expected in `Surprise` with user, item, and rating information.  Then, compare the performance of at least the `KNNBasic`, `SVD`, `NMF`, `SlopeOne`, and `CoClustering` algorithms to build your recommendations.  For more information on the algorithms see the documentation for the algorithm package [here](https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html).

Share the results of your investigation and include the results of your cross validation and a basic description of your dataset with your peers.


In [20]:
# define a rating scale at 0.5 intervals
reader = Reader(rating_scale=(0.5, 5))
sf= Dataset.load_from_df(movie_reviews[["userId","movieId","rating"]], reader)
sf


In [21]:
# Train and test set
train= sf.build_full_trainset()
test= train.build_testset()

In [22]:
#Build the model
model = SVD(n_epochs=10000, biased=False , n_factors=2)

In [23]:
# Fit the SVD model
model.fit(train)

# Test set
predictions_list=model.test(test)
predictions_list[:5]


[Prediction(uid=1, iid=1, r_ui=4.0, est=4.856162830186929, details={'was_impossible': False}),
 Prediction(uid=1, iid=3, r_ui=4.0, est=3.968257656203827, details={'was_impossible': False}),
 Prediction(uid=1, iid=6, r_ui=4.0, est=4.843926697792446, details={'was_impossible': False}),
 Prediction(uid=1, iid=47, r_ui=5.0, est=4.991452041305833, details={'was_impossible': False}),
 Prediction(uid=1, iid=50, r_ui=5.0, est=5, details={'was_impossible': False})]

In [27]:
len(predictions_list)

100836

In [33]:
 # prepare model list of `KNNBasic`, `SVD`, `NMF`, `SlopeOne`, and `CoClustering` as name value pair
models = {
    "KNNBasic": KNNBasic(),
    "SVD": SVD(),
    "NMF": NMF(),
    "SlopeOne": SlopeOne(),
    "CoClustering": CoClustering()
}
models.items()

dict_items([('KNNBasic', <surprise.prediction_algorithms.knns.KNNBasic object at 0x7ddfa6f01c10>), ('SVD', <surprise.prediction_algorithms.matrix_factorization.SVD object at 0x7ddfa6f01f70>), ('NMF', <surprise.prediction_algorithms.matrix_factorization.NMF object at 0x7ddfa6f02270>), ('SlopeOne', <surprise.prediction_algorithms.slope_one.SlopeOne object at 0x7ddfa6f03410>), ('CoClustering', <surprise.prediction_algorithms.co_clustering.CoClustering object at 0x7ddfa6f02360>)])

In [ ]:
# Perform cross validation on the various models and the best model on basis of minimum MSE

In [51]:
cross_val_results = cross_validate(model,sf,cv=10, measures=['MSE'])

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.


In [55]:
# Find mean of values inside this tuple
cross_val_results['test_mse'].mean()

0.8860599127877904

In [56]:
from datetime import datetime
from timeit import default_timer as timer
best_models = {}
results_cv = []
for name, model in models.items():

    start_time = timer()
    print(f"Training model and finding score for {name}...at ", datetime.now(),  " UTC")
    model.fit(train)
    train_time_model = timer() - start_time

    # Compute cross validation metrics on train and test sets
    # Store results
    cross_val_results = cross_validate(model, sf, measures=['MSE'])
    print(f"cv results score for {name}...at ", datetime.now(),  " UTC", cross_val_results)
    results_cv.append({"model": f" {name}",
                       "best_mse_cv": cross_val_results['test_mse'].mean(),
                       "Train Time": train_time_model})


Training model and finding score for KNNBasic...at  2026-05-06 16:09:51.448595  UTC
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
cv results score for KNNBasic...at  2026-05-06 16:09:59.294321  UTC {'test_mse': array([0.88581649, 0.90734598, 0.89832472, 0.89009018, 0.90700977]), 'fit_time': (0.10718345642089844, 0.11399173736572266, 0.11176896095275879, 0.12065505981445312, 0.16787147521972656), 'test_time': (1.1858642101287842, 1.2507071495056152, 1.066317081451416, 1.0446290969848633, 1.3667433261871338)}
Training model and finding score for SVD...at  2026-05-06 16:09:59.294863  UTC
cv results score

In [57]:
results_cv_df = pd.DataFrame(results_cv).sort_values("best_mse_cv", ascending=False).reset_index(drop=True)
results_cv_df.sort_values(by="best_mse_cv", ascending=False, inplace=True)
results_cv_df


,model,best_mse_cv,Train Time
0,KNNBasic,0.897717,0.156158
1,CoClustering,0.893513,2.024368
2,NMF,0.853777,1.866451
3,SlopeOne,0.813685,6.871221
4,SVD,0.764855,1.023933
